# GeoDiff-GAN on Kaggle

This notebook prepares Sentinel-2 L2A RGB patches, quarantines black edge/corner
artifacts, visualizes accepted and rejected data, trains GeoDiff-GAN, and exports
diagnostics. Cells that touch data reconstruct their paths so they can be rerun
after a Kaggle kernel restart.

`FAST_DEV_RUN=True` verifies the pipeline only. A one-tile run cannot measure
geographic generalization or train a publishable super-resolution model.


In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

def run(command, cwd=None):
    """Run a command with visible, copyable output."""
    command = list(map(str, command))
    project_modules = {
        "geodiff-prepare": "geodiff_gan.cli.prepare_sentinel",
        "geodiff-caption": "geodiff_gan.cli.caption_qwen",
        "geodiff-train": "geodiff_gan.cli.train",
        "geodiff-infer": "geodiff_gan.cli.infer",
        "geodiff-evaluate": "geodiff_gan.cli.evaluate",
        "geodiff-baselines": "geodiff_gan.cli.baselines",
        "geodiff-debug": "geodiff_gan.cli.debug",
    }
    if command and command[0] in project_modules and shutil.which(command[0]) is None:
        command = [sys.executable, "-m", project_modules[command[0]], *command[1:]]
    print("+", " ".join(command), flush=True)
    subprocess.run(command, cwd=cwd, check=True)

REPOSITORY_URL = "https://github.com/shashankjs2002/SI-SR-1.git"
REPOSITORY_DIR = Path("/kaggle/working/geodiff-gan-sentinel2")
SENTINEL_INPUT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/geodiff-gan-output")

FAST_DEV_RUN = True
# Limit SAFE products only for pipeline testing. Use None for research runs.
MAX_TILES = 1 if FAST_DEV_RUN else None
RUN_DATA_PREPARATION = True
RUN_CAPTIONING = False
CAPTION_LIMIT_PER_SPLIT = 200 if FAST_DEV_RUN else None
STAGES_TO_RUN = ["base", "vae", "diffusion", "joint"]
# Add "edit" only after generating meaningful captions.

PATCH_SIZE = 512
PATCH_STRIDE = 384
MINIMUM_VALID_FRACTION = 0.95

# The calibrated preset keeps synthetic noise below the image signal.
DEGRADATION_SEVERITY = "mild"
DEGRADATION_SEED = 42

# Quarantine only obvious no-data borders/corners; do not delete source patches.
FILTER_EDGE_PATCHES = True
BLACK_THRESHOLD = 1e-6
MAX_EDGE_BLACK_FRACTION = 0.002
MAX_CORNER_BLACK_FRACTION = 0.002
MAX_OVERALL_BLACK_FRACTION = 0.005
EDGE_WIDTH = 32
CORNER_SIZE = 64
RANDOM_VISUALIZATION_SEED = 42
VISUALIZE_ACCEPTED = 5
VISUALIZE_REJECTED = 5

WORK_ROOT.mkdir(parents=True, exist_ok=True)
print("Work root:", WORK_ROOT)


## 1. Clone, update, and install

The cell updates an existing clone with a fast-forward pull, verifies the package
files, installs Kaggle dependencies, and installs the project in editable mode.
Push the latest local commits before running this notebook on Kaggle.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPOSITORY_URL = globals().get(
    "REPOSITORY_URL", "https://github.com/shashankjs2002/SI-SR-1.git"
)
REPOSITORY_DIR = globals().get(
    "REPOSITORY_DIR", Path("/kaggle/working/geodiff-gan-sentinel2")
)

def run(command, cwd=None):
    command = list(map(str, command))
    environment = os.environ.copy()
    source_root = REPOSITORY_DIR / "src"
    if source_root.exists():
        existing_pythonpath = environment.get("PYTHONPATH", "")
        environment["PYTHONPATH"] = os.pathsep.join(
            part for part in (str(source_root.resolve()), existing_pythonpath) if part
        )
    print("+", " ".join(command), flush=True)
    subprocess.run(command, cwd=cwd, check=True, env=environment)

if REPOSITORY_DIR.exists():
    if not (REPOSITORY_DIR / ".git").exists():
        raise RuntimeError(
            f"{REPOSITORY_DIR} exists but is not a Git clone. "
            "Rename it or start a fresh Kaggle session."
        )
    print(f"Updating existing clone: {REPOSITORY_DIR}")
    run(["git", "pull", "--ff-only"], cwd=REPOSITORY_DIR)
else:
    run(["git", "clone", "--depth", "1", REPOSITORY_URL, REPOSITORY_DIR])

required = [
    REPOSITORY_DIR / "pyproject.toml",
    REPOSITORY_DIR / "requirements-kaggle.txt",
    REPOSITORY_DIR / "src" / "geodiff_gan",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(
        "The GitHub repository is missing required project files:\n- "
        + "\n- ".join(missing)
    )

run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"],
    cwd=REPOSITORY_DIR,
)
run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"],
    cwd=REPOSITORY_DIR,
)
os.chdir(REPOSITORY_DIR)
source_root = (REPOSITORY_DIR / "src").resolve()
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

import importlib
importlib.invalidate_caches()
import geodiff_gan

print("Repository:", Path.cwd())
print("Kernel package:", Path(geodiff_gan.__file__).resolve())
run([
    sys.executable,
    "-c",
    "import geodiff_gan; print('Subprocess package:', geodiff_gan.__file__)",
], cwd=REPOSITORY_DIR)
run(["git", "log", "-1", "--oneline"], cwd=REPOSITORY_DIR)


In [ ]:
import torch

GPU_COUNT = torch.cuda.device_count()
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", GPU_COUNT)
for index in range(GPU_COUNT):
    properties = torch.cuda.get_device_properties(index)
    print(index, properties.name, f"{properties.total_memory / 2**30:.1f} GB")
if GPU_COUNT == 0:
    raise RuntimeError("Enable a Kaggle GPU accelerator before training.")


## 2. Prepare Sentinel-2 patches

This cell is restart-safe: it recreates all path/configuration variables if the
first cell is no longer in memory. The current extractor maps each 10 m RGB window
into the 20 m SCL coordinate grid before reading the mask, avoiding out-of-range
SCL reads at the right and bottom of a tile.


In [ ]:
from pathlib import Path
from collections import Counter
import importlib.util
import inspect
import json
import subprocess
import sys

SENTINEL_INPUT = globals().get("SENTINEL_INPUT", Path("/kaggle/input"))
WORK_ROOT = globals().get("WORK_ROOT", Path("/kaggle/working/geodiff-gan-output"))
REPOSITORY_DIR = globals().get(
    "REPOSITORY_DIR", Path("/kaggle/working/geodiff-gan-sentinel2")
)
RUN_DATA_PREPARATION = globals().get("RUN_DATA_PREPARATION", True)
FAST_DEV_RUN = globals().get("FAST_DEV_RUN", True)
MAX_TILES = globals().get("MAX_TILES", 1 if FAST_DEV_RUN else None)
PATCH_SIZE = globals().get("PATCH_SIZE", 512)
PATCH_STRIDE = globals().get("PATCH_STRIDE", 384)
MINIMUM_VALID_FRACTION = globals().get("MINIMUM_VALID_FRACTION", 0.95)

if "run" not in globals():
    def run(command, cwd=None):
        command = list(map(str, command))
        print("+", " ".join(command), flush=True)
        subprocess.run(command, cwd=cwd, check=True)

all_safe_products = sorted(SENTINEL_INPUT.rglob("*.SAFE"))
safe_products = (
    all_safe_products[:MAX_TILES] if MAX_TILES is not None else all_safe_products
)
print(f"Found {len(all_safe_products)} SAFE products")
print(f"Selected {len(safe_products)} SAFE products (MAX_TILES={MAX_TILES})")
for product in safe_products[:10]:
    print(" -", product)

if importlib.util.find_spec("geodiff_gan") is None:
    source_root = REPOSITORY_DIR / "src"
    if source_root.exists():
        sys.path.insert(0, str(source_root))
    if importlib.util.find_spec("geodiff_gan") is None:
        raise RuntimeError(
            "GeoDiff-GAN is not installed in this Kaggle session. "
            "Run the '1. Clone, update, and install' cell first."
        )

from geodiff_gan.data import sentinel as sentinel_module

extractor_source = inspect.getsource(sentinel_module.extract_product_patches)
if "from_bounds" not in extractor_source or "boundless=True" not in extractor_source:
    raise RuntimeError(
        "The installed extractor predates the 10 m RGB to 20 m SCL window fix. "
        "Push the latest repository commits, rerun the install cell, and retry."
    )
print("Extractor:", Path(sentinel_module.__file__).resolve())

PATCH_ROOT = WORK_ROOT / "patches"
MANIFEST = WORK_ROOT / "manifest.jsonl"
if MANIFEST.exists() and MAX_TILES is not None:
    existing_records = [
        json.loads(line)
        for line in MANIFEST.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    existing_tiles = {record["tile_id"] for record in existing_records}
    if len(existing_tiles) > MAX_TILES:
        raise RuntimeError(
            f"Existing manifest contains {len(existing_tiles)} tiles, but MAX_TILES="
            f"{MAX_TILES}. Delete {MANIFEST} and {PATCH_ROOT}, or use a fresh WORK_ROOT."
        )
if RUN_DATA_PREPARATION and not MANIFEST.exists():
    if not safe_products:
        raise FileNotFoundError(
            "No extracted .SAFE directory was found below /kaggle/input. "
            "Attach a dataset containing Sentinel-2 L2A SAFE directories."
        )
    prepare_command = [
        sys.executable, "-m", "geodiff_gan.cli.prepare_sentinel",
        "--input", SENTINEL_INPUT,
        "--output", PATCH_ROOT,
        "--manifest", MANIFEST,
        "--patch-size", PATCH_SIZE,
        "--stride", PATCH_STRIDE,
        "--minimum-valid-fraction", MINIMUM_VALID_FRACTION,
    ]
    if MAX_TILES is not None:
        prepare_command.extend(["--max-products", MAX_TILES])
    run(prepare_command)
elif MANIFEST.exists():
    print("Reusing existing manifest:", MANIFEST)

if not MANIFEST.exists():
    raise FileNotFoundError(f"Manifest not found: {MANIFEST}")
records = [
    json.loads(line)
    for line in MANIFEST.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
if not records:
    raise RuntimeError(
        "Data preparation produced no valid patches. Inspect the SAFE product, "
        "SCL masks, and MINIMUM_VALID_FRACTION before training."
    )
split_counts = Counter(record["split"] for record in records)
tile_counts = Counter(record["tile_id"] for record in records)
print("Patches:", len(records))
print("Splits:", split_counts)
print("Tiles:", tile_counts)
missing_splits = [name for name in ("train", "val", "test") if not split_counts[name]]
if missing_splits:
    message = (
        f"Missing tile-level splits: {missing_splits}. Found {len(tile_counts)} unique "
        "MGRS tile(s). A one-tile dataset is suitable only for FAST_DEV_RUN pipeline "
        "debugging; it cannot measure geographic generalization."
    )
    if not FAST_DEV_RUN:
        raise RuntimeError(message + " Attach more geographically separated SAFE products.")
    print("WARNING:", message)


## 3. Quarantine black edge/corner patches

The preparation mask already removes most invalid windows. This additional pass
targets obvious zero-filled no-data borders. It never deletes data:

- saves the first unfiltered manifest as `manifest.before-edge-filter.jsonl`;
- moves rejected patches into `quarantine_edge_patches/`;
- writes their paths, reasons, and fractions to `rejected-edge-patches.jsonl`;
- rewrites the active manifest with accepted patches only.

The cell is idempotent and can be rerun safely.


In [ ]:
from pathlib import Path
from collections import Counter
import json
import shutil
import numpy as np

WORK_ROOT = globals().get("WORK_ROOT", Path("/kaggle/working/geodiff-gan-output"))
MANIFEST = globals().get("MANIFEST", WORK_ROOT / "manifest.jsonl")
FILTER_EDGE_PATCHES = globals().get("FILTER_EDGE_PATCHES", True)
BLACK_THRESHOLD = globals().get("BLACK_THRESHOLD", 1e-6)
MAX_EDGE_BLACK_FRACTION = globals().get("MAX_EDGE_BLACK_FRACTION", 0.002)
MAX_CORNER_BLACK_FRACTION = globals().get("MAX_CORNER_BLACK_FRACTION", 0.002)
MAX_OVERALL_BLACK_FRACTION = globals().get("MAX_OVERALL_BLACK_FRACTION", 0.005)
EDGE_WIDTH = globals().get("EDGE_WIDTH", 32)
CORNER_SIZE = globals().get("CORNER_SIZE", 64)

BACKUP_MANIFEST = WORK_ROOT / "manifest.before-edge-filter.jsonl"
REJECTED_MANIFEST = WORK_ROOT / "rejected-edge-patches.jsonl"
QUARANTINE_ROOT = WORK_ROOT / "quarantine_edge_patches"

if not MANIFEST.exists():
    raise FileNotFoundError(f"Manifest not found: {MANIFEST}")
active_records = [
    json.loads(line)
    for line in MANIFEST.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

if not BACKUP_MANIFEST.exists():
    shutil.copy2(MANIFEST, BACKUP_MANIFEST)
    print("Saved original manifest:", BACKUP_MANIFEST)

existing_rejected = []
if REJECTED_MANIFEST.exists():
    existing_rejected = [
        json.loads(line)
        for line in REJECTED_MANIFEST.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
rejected_keys = {
    record.get("original_patch", record.get("patch"))
    for record in existing_rejected
}

def inspect_black_background(path):
    path = Path(path)
    if not path.exists():
        return True, {
            "reason": "missing_patch",
            "overall_black_fraction": 1.0,
            "edge_black_fraction": 1.0,
            "corner_black_fraction": 1.0,
        }
    try:
        with np.load(path) as data:
            hr = np.asarray(data["hr"], dtype=np.float32)
    except Exception as error:
        return True, {
            "reason": f"unreadable_patch:{type(error).__name__}",
            "overall_black_fraction": 1.0,
            "edge_black_fraction": 1.0,
            "corner_black_fraction": 1.0,
        }
    if hr.ndim != 3 or hr.shape[0] != 3:
        return True, {
            "reason": f"unexpected_shape:{tuple(hr.shape)}",
            "overall_black_fraction": 1.0,
            "edge_black_fraction": 1.0,
            "corner_black_fraction": 1.0,
        }
    if not np.isfinite(hr).all():
        return True, {
            "reason": "nonfinite_pixels",
            "overall_black_fraction": 1.0,
            "edge_black_fraction": 1.0,
            "corner_black_fraction": 1.0,
        }

    black = np.all(hr <= BLACK_THRESHOLD, axis=0)
    height, width = black.shape
    edge = min(EDGE_WIDTH, height // 2, width // 2)
    corner = min(CORNER_SIZE, height // 2, width // 2)
    edge_pixels = np.concatenate([
        black[:edge].ravel(),
        black[-edge:].ravel(),
        black[edge:-edge, :edge].ravel(),
        black[edge:-edge, -edge:].ravel(),
    ])
    corner_pixels = np.concatenate([
        black[:corner, :corner].ravel(),
        black[:corner, -corner:].ravel(),
        black[-corner:, :corner].ravel(),
        black[-corner:, -corner:].ravel(),
    ])
    stats = {
        "overall_black_fraction": float(black.mean()),
        "edge_black_fraction": float(edge_pixels.mean()),
        "corner_black_fraction": float(corner_pixels.mean()),
    }
    reasons = []
    if stats["overall_black_fraction"] > MAX_OVERALL_BLACK_FRACTION:
        reasons.append("overall_black")
    if stats["edge_black_fraction"] > MAX_EDGE_BLACK_FRACTION:
        reasons.append("edge_black")
    if stats["corner_black_fraction"] > MAX_CORNER_BLACK_FRACTION:
        reasons.append("corner_black")
    stats["reason"] = ",".join(reasons) if reasons else "accepted"
    return bool(reasons), stats

kept_records = []
newly_rejected = []
if FILTER_EDGE_PATCHES:
    for index, record in enumerate(active_records, start=1):
        original_patch = record["patch"]
        reject, stats = inspect_black_background(original_patch)
        if reject:
            rejected_record = dict(record)
            rejected_record["original_patch"] = original_patch
            rejected_record["filter"] = stats
            source_path = Path(original_patch)
            if source_path.exists():
                destination = QUARANTINE_ROOT / record["tile_id"] / source_path.name
                destination.parent.mkdir(parents=True, exist_ok=True)
                if source_path.resolve() != destination.resolve():
                    shutil.move(str(source_path), str(destination))
                rejected_record["patch"] = str(destination)
            if original_patch not in rejected_keys:
                newly_rejected.append(rejected_record)
                rejected_keys.add(original_patch)
        else:
            kept_records.append(record)
        if index % 200 == 0 or index == len(active_records):
            print(f"Scanned {index}/{len(active_records)} patches")
else:
    kept_records = active_records
    print("Edge/corner filtering disabled.")

all_rejected = existing_rejected + newly_rejected
MANIFEST.write_text(
    "".join(json.dumps(record) + "\n" for record in kept_records),
    encoding="utf-8",
)
REJECTED_MANIFEST.write_text(
    "".join(json.dumps(record) + "\n" for record in all_rejected),
    encoding="utf-8",
)

records = kept_records
split_counts = Counter(record["split"] for record in records)
tile_counts = Counter(record["tile_id"] for record in records)
print("Accepted patches:", len(records))
print("Newly quarantined:", len(newly_rejected))
print("Total quarantined:", len(all_rejected))
print("Accepted splits:", split_counts)
print("Rejected manifest:", REJECTED_MANIFEST)
print("Quarantine:", QUARANTINE_ROOT)

if not records:
    raise RuntimeError(
        "The edge/corner filter rejected every patch. Relax the thresholds only "
        "after inspecting the rejected-sample visualization below."
    )


## 4. Visualize five accepted samples and their LR degradation

Each row uses the same degradation implementation as training:

1. native 10 m HR target;
2. clean MTF-blurred and 4x area-downsampled 40 m LR;
3. observed LR after the calibrated `mild` noise/quantization preset;
4. bicubic enlargement for visual comparison only.

The title reports `mean(|observed-clean|) / mean(|clean|)`. If this ratio is large,
stop and inspect the degradation configuration before training.


In [ ]:
from pathlib import Path
import json
import random
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.nn import functional as F

from geodiff_gan.models.degradation import random_degradation

WORK_ROOT = globals().get("WORK_ROOT", Path("/kaggle/working/geodiff-gan-output"))
MANIFEST = globals().get("MANIFEST", WORK_ROOT / "manifest.jsonl")
DEGRADATION_SEVERITY = globals().get("DEGRADATION_SEVERITY", "mild")
DEGRADATION_SEED = globals().get("DEGRADATION_SEED", 42)
RANDOM_VISUALIZATION_SEED = globals().get("RANDOM_VISUALIZATION_SEED", 42)
VISUALIZE_ACCEPTED = globals().get("VISUALIZE_ACCEPTED", 5)

accepted_records = [
    json.loads(line)
    for line in MANIFEST.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
randomizer = random.Random(RANDOM_VISUALIZATION_SEED)
selected = randomizer.sample(
    accepted_records, k=min(VISUALIZE_ACCEPTED, len(accepted_records))
)

def chw_to_rgb(tensor):
    return tensor.detach().cpu().clamp(0, 1).permute(1, 2, 0).numpy()

figure, axes = plt.subplots(
    len(selected), 4, figsize=(16, 4 * len(selected)), squeeze=False
)
for row, record in enumerate(selected):
    with np.load(record["patch"]) as data:
        hr = torch.from_numpy(data["hr"]).float()
    if hr.ndim == 3 and hr.shape[-1] == 3:
        hr = hr.permute(2, 0, 1)
    generator = torch.Generator().manual_seed(DEGRADATION_SEED + row)
    observed_lr, parameters, clean_lr = random_degradation(
        hr.unsqueeze(0),
        scale=4,
        generator=generator,
        return_clean=True,
        severity=DEGRADATION_SEVERITY,
    )
    bicubic = F.interpolate(
        observed_lr,
        size=hr.shape[-2:],
        mode="bicubic",
        align_corners=False,
    ).clamp(0, 1)
    noise_l1 = (observed_lr - clean_lr).abs().mean()
    noise_to_signal = noise_l1 / clean_lr.abs().mean().clamp_min(1e-8)
    images = [hr, clean_lr[0], observed_lr[0], bicubic[0]]
    titles = [
        f"HR 10 m\n{record['tile_id']} r{record['row']} c{record['col']}",
        "Clean LR 40 m",
        (
            f"Observed LR ({DEGRADATION_SEVERITY})\n"
            f"noise/signal={float(noise_to_signal):.4f}"
        ),
        f"Bicubic 4x\nparams={parameters[0].tolist()}",
    ]
    for column, (image, title) in enumerate(zip(images, titles)):
        axes[row, column].imshow(chw_to_rgb(image))
        axes[row, column].set_title(title, fontsize=10)
        axes[row, column].axis("off")
figure.tight_layout()
plt.show()


## 5. Visualize quarantined samples

Red pixels identify zero-filled RGB background. The figure reads directly from
`rejected-edge-patches.jsonl`, so it shows the exact patches removed from training.
If no sample appears, the current tile had no patch above the configured thresholds.


In [ ]:
from pathlib import Path
import json
import random
import matplotlib.pyplot as plt
import numpy as np

WORK_ROOT = globals().get("WORK_ROOT", Path("/kaggle/working/geodiff-gan-output"))
REJECTED_MANIFEST = globals().get(
    "REJECTED_MANIFEST", WORK_ROOT / "rejected-edge-patches.jsonl"
)
BLACK_THRESHOLD = globals().get("BLACK_THRESHOLD", 1e-6)
RANDOM_VISUALIZATION_SEED = globals().get("RANDOM_VISUALIZATION_SEED", 42)
VISUALIZE_REJECTED = globals().get("VISUALIZE_REJECTED", 5)

rejected_records = []
if REJECTED_MANIFEST.exists():
    rejected_records = [
        json.loads(line)
        for line in REJECTED_MANIFEST.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
viewable = [record for record in rejected_records if Path(record["patch"]).exists()]

if not viewable:
    print("No quarantined patch files are available to visualize.")
else:
    randomizer = random.Random(RANDOM_VISUALIZATION_SEED)
    selected_rejected = randomizer.sample(
        viewable, k=min(VISUALIZE_REJECTED, len(viewable))
    )
    figure, axes = plt.subplots(
        len(selected_rejected),
        4,
        figsize=(16, 4 * len(selected_rejected)),
        squeeze=False,
    )
    for row, record in enumerate(selected_rejected):
        with np.load(record["patch"]) as data:
            hr = np.asarray(data["hr"], dtype=np.float32)
            valid_mask = (
                np.asarray(data["valid_mask"]).astype(bool)
                if "valid_mask" in data
                else np.ones(hr.shape[-2:], dtype=bool)
            )
        rgb = np.clip(np.moveaxis(hr, 0, -1), 0, 1)
        black = np.all(hr <= BLACK_THRESHOLD, axis=0)
        overlay = rgb.copy()
        overlay[black] = np.array([1.0, 0.0, 0.0], dtype=np.float32)
        filter_stats = record.get("filter", {})
        displays = [
            (rgb, "Rejected HR"),
            (black, "Black-background mask"),
            (overlay, "Black pixels marked red"),
            (~valid_mask, "Original invalid/SCL mask"),
        ]
        for column, (image, title) in enumerate(displays):
            axes[row, column].imshow(
                image,
                cmap="gray" if image.ndim == 2 else None,
                vmin=0 if image.ndim == 2 else None,
                vmax=1 if image.ndim == 2 else None,
            )
            suffix = ""
            if column == 0:
                suffix = (
                    f"\n{filter_stats.get('reason', 'unknown')}"
                    f" | edge={filter_stats.get('edge_black_fraction', float('nan')):.4f}"
                    f" | corner={filter_stats.get('corner_black_fraction', float('nan')):.4f}"
                )
            axes[row, column].set_title(title + suffix, fontsize=10)
            axes[row, column].axis("off")
    figure.tight_layout()
    plt.show()
    print(f"Showing {len(selected_rejected)} of {len(rejected_records)} rejected records.")


## 6. Optional Qwen3-VL-8B captions

Captioning is offline preprocessing. Keep it disabled for pipeline debugging and
enable it only when edit-mode prompt conditioning is required.


In [ ]:
from pathlib import Path
import subprocess
import sys

WORK_ROOT = globals().get("WORK_ROOT", Path("/kaggle/working/geodiff-gan-output"))
MANIFEST = globals().get("MANIFEST", WORK_ROOT / "manifest.jsonl")
RUN_CAPTIONING = globals().get("RUN_CAPTIONING", False)
CAPTION_LIMIT_PER_SPLIT = globals().get("CAPTION_LIMIT_PER_SPLIT", 200)
CAPTIONS = WORK_ROOT / "captions.jsonl"

if "run" not in globals():
    def run(command, cwd=None):
        command = list(map(str, command))
        print("+", " ".join(command), flush=True)
        subprocess.run(command, cwd=cwd, check=True)

if RUN_CAPTIONING:
    for split in ("train", "val", "test"):
        command = [
            sys.executable, "-m", "geodiff_gan.cli.caption_qwen",
            "--manifest", MANIFEST,
            "--output", CAPTIONS,
            "--model", "Qwen/Qwen3-VL-8B-Instruct",
            "--split", split,
        ]
        if CAPTION_LIMIT_PER_SPLIT is not None:
            command.extend(["--limit", CAPTION_LIMIT_PER_SPLIT])
        run(command)
else:
    print("Captioning skipped. Prompt conditioning will use null prompts.")
print("Caption file:", CAPTIONS if CAPTIONS.exists() else None)


## 7. Build Kaggle runtime configs


In [ ]:
from pathlib import Path
import copy
import torch
import yaml

REPOSITORY_DIR = globals().get(
    "REPOSITORY_DIR", Path("/kaggle/working/geodiff-gan-sentinel2")
)
WORK_ROOT = globals().get("WORK_ROOT", Path("/kaggle/working/geodiff-gan-output"))
MANIFEST = globals().get("MANIFEST", WORK_ROOT / "manifest.jsonl")
CAPTIONS = globals().get("CAPTIONS", WORK_ROOT / "captions.jsonl")
FAST_DEV_RUN = globals().get("FAST_DEV_RUN", True)
DEGRADATION_SEVERITY = globals().get("DEGRADATION_SEVERITY", "mild")
DEGRADATION_SEED = globals().get("DEGRADATION_SEED", 42)
GPU_COUNT = globals().get("GPU_COUNT", torch.cuda.device_count())

with (REPOSITORY_DIR / "configs/default.yaml").open() as handle:
    runtime_config = yaml.safe_load(handle)

runtime_config["data"]["manifest"] = str(MANIFEST)
runtime_config["data"]["captions"] = str(CAPTIONS) if CAPTIONS.exists() else None
runtime_config["data"]["degradation_severity"] = DEGRADATION_SEVERITY
runtime_config["data"]["degradation_seed"] = DEGRADATION_SEED
runtime_config["training"]["batch_size"] = 1
runtime_config["training"]["gradient_accumulation"] = max(1, 16 // max(GPU_COUNT, 1))
runtime_config["training"]["num_workers"] = 2
runtime_config["training"]["amp"] = True
runtime_config["training"]["gradient_checkpointing"] = True

# Persist tensor diagnostics during training; smoke runs capture every step.
runtime_config.setdefault("debug", {})
runtime_config["debug"].update({
    "enabled": True,
    "output_dir": str(WORK_ROOT / "training_debug"),
    "every_n_steps": 1 if FAST_DEV_RUN else 100,
    "max_exports_per_epoch": 2 if FAST_DEV_RUN else 5,
    "print_tensor_stats": True,
    "fail_on_nonfinite": True,
    "panel_size": 280 if FAST_DEV_RUN else 360,
    "histogram_bins": 48 if FAST_DEV_RUN else 64,
    "max_feature_channels": 12 if FAST_DEV_RUN else 16,
    "save_tensors": FAST_DEV_RUN,
    "max_saved_tensors": 24,
})

if FAST_DEV_RUN:
    with (REPOSITORY_DIR / "configs/smoke.yaml").open() as handle:
        smoke = yaml.safe_load(handle)
    runtime_config["model"].update(smoke["model"])
    runtime_config["text_encoder"] = smoke["text_encoder"]
    runtime_config["training"].update(smoke["training"])
    runtime_config["training"]["batch_size"] = 1
    runtime_config["training"]["gradient_accumulation"] = 1

EPOCHS = (
    {"base": 1, "vae": 1, "diffusion": 1, "joint": 1, "edit": 1}
    if FAST_DEV_RUN
    else {"base": 20, "vae": 20, "diffusion": 80, "joint": 20, "edit": 10}
)
CONFIG_ROOT = WORK_ROOT / "configs"
RUN_ROOT = WORK_ROOT / "runs"
CONFIG_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print("Degradation severity:", runtime_config["data"]["degradation_severity"])
print("Degradation seed:", runtime_config["data"]["degradation_seed"])
print(yaml.safe_dump(runtime_config, sort_keys=False)[:3500])


## 8. Train selected stages


In [ ]:
def train_stage(stage, initial_checkpoint=None):
    config = copy.deepcopy(runtime_config)
    output_dir = RUN_ROOT / stage
    config["training"]["stage"] = stage
    config["training"]["epochs"] = EPOCHS[stage]
    config["training"]["output_dir"] = str(output_dir)
    config["training"]["init_checkpoint"] = (
        str(initial_checkpoint) if initial_checkpoint is not None else None
    )
    config["training"]["resume"] = None
    if stage == "edit":
        config["training"]["learning_rate"] = 1e-5
        config["training"]["loss_weights"]["consistency"] = 0.2
    elif stage == "joint":
        config["training"]["learning_rate"] = 2e-5
    config_path = CONFIG_ROOT / f"{stage}.yaml"
    with config_path.open("w") as handle:
        yaml.safe_dump(config, handle, sort_keys=False)

    module_command = ["-m", "geodiff_gan.cli.train", "--config", config_path]
    if GPU_COUNT > 1:
        command = ["torchrun", "--standalone", f"--nproc_per_node={GPU_COUNT}", *module_command]
    else:
        command = [sys.executable, *module_command]
    run(command, cwd=REPOSITORY_DIR)
    checkpoints = sorted(output_dir.glob(f"{stage}_epoch_*.pt"))
    if not checkpoints:
        raise RuntimeError(f"No checkpoint produced for stage {stage}")
    return checkpoints[-1]

if "edit" in STAGES_TO_RUN and not CAPTIONS.exists():
    raise RuntimeError("Generate Qwen3-VL captions before training edit mode.")

CHECKPOINTS = {}
previous_checkpoint = None
for stage in STAGES_TO_RUN:
    previous_checkpoint = train_stage(stage, previous_checkpoint)
    CHECKPOINTS[stage] = previous_checkpoint
    print(stage, "->", previous_checkpoint)

from IPython.display import Image as DisplayImage, display
for stage in STAGES_TO_RUN:
    curve = RUN_ROOT / stage / "training_curves.png"
    if curve.exists():
        print(f"Training curves: {stage}")
        display(DisplayImage(filename=str(curve)))
CHECKPOINTS


## 9. Evaluate stochastic SR and baselines

Evaluation uses deterministic LR degradations for each patch, reports observed-LR
noise diagnostics, and projects SR output against the clean sensor-degraded LR
rather than fitting sampled noise.


In [ ]:
from collections import Counter
from pathlib import Path
import json
import sys

if "records" not in globals():
    records = [
        json.loads(line)
        for line in MANIFEST.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
split_counts = Counter(record["split"] for record in records)
evaluation_split = "test" if split_counts["test"] else (
    "val" if split_counts["val"] else "train"
)
evaluation_checkpoint = CHECKPOINTS.get("joint", previous_checkpoint)
evaluation_config = CONFIG_ROOT / f"{STAGES_TO_RUN[-1]}.yaml"
EVAL_ROOT = WORK_ROOT / "evaluation"

run([
    sys.executable, "-m", "geodiff_gan.cli.evaluate",
    "--config", evaluation_config,
    "--checkpoint", evaluation_checkpoint,
    "--output", EVAL_ROOT,
    "--split", evaluation_split,
    "--samples", 2 if FAST_DEV_RUN else 8,
    "--steps", 2 if FAST_DEV_RUN else 20,
    "--mode", "sr",
    "--limit", 4 if FAST_DEV_RUN else 100,
])

base_checkpoint = CHECKPOINTS.get("base")
baseline_command = [
    sys.executable, "-m", "geodiff_gan.cli.baselines",
    "--config", evaluation_config,
    "--output", WORK_ROOT / "baselines.json",
    "--split", evaluation_split,
    "--limit", 4 if FAST_DEV_RUN else 100,
]
if base_checkpoint is not None:
    baseline_command.extend(["--base-checkpoint", base_checkpoint])
run(baseline_command)
print((EVAL_ROOT / "metrics.json").read_text(encoding="utf-8"))
print((WORK_ROOT / "baselines.json").read_text(encoding="utf-8"))


## 10. Generate visual diagnostics


In [ ]:
from IPython.display import Image as DisplayImage, display

DEBUG_ROOT = WORK_ROOT / "debug_patch_0"
run([
    sys.executable, "-m", "geodiff_gan.cli.debug",
    "--config", evaluation_config,
    "--checkpoint", evaluation_checkpoint,
    "--output", DEBUG_ROOT,
    "--split", evaluation_split,
    "--index", 0,
    "--mode", "sr",
    "--steps", 2 if FAST_DEV_RUN else 20,
    "--diffusion-every", 1 if FAST_DEV_RUN else 5,
    "--save-tensors",
])

for diagnostic_image in (
    "overview.png",
    "stage_intermediates.png",
    "features.png",
    "tensor_histograms.png",
    "frequency_spectra.png",
    "policy_overlays.png",
    "edges_and_wavelets.png",
    "diffusion_trajectory.png",
    "projection_trajectory.png",
    "loss_breakdown.png",
):
    path = DEBUG_ROOT / diagnostic_image
    if path.exists():
        display(DisplayImage(filename=str(path)))
print((DEBUG_ROOT / "summary.txt").read_text())


## 11. Export artifacts


In [ ]:
from pathlib import Path
import shutil

EXPORT_ROOT = Path("/kaggle/working/geodiff-gan-results")
if EXPORT_ROOT.exists():
    shutil.rmtree(EXPORT_ROOT)
EXPORT_ROOT.mkdir(parents=True)
for name in (
    "configs",
    "runs",
    "evaluation",
    "debug_patch_0",
    "training_debug",
    "quarantine_edge_patches",
):
    source_path = WORK_ROOT / name
    if source_path.exists():
        shutil.copytree(source_path, EXPORT_ROOT / name)
for file_name in (
    "manifest.jsonl",
    "manifest.before-edge-filter.jsonl",
    "rejected-edge-patches.jsonl",
    "captions.jsonl",
    "baselines.json",
):
    source_path = WORK_ROOT / file_name
    if source_path.exists():
        shutil.copy2(source_path, EXPORT_ROOT / file_name)
archive = shutil.make_archive("/kaggle/working/geodiff-gan-results", "zip", EXPORT_ROOT)
print("Exported:", archive)
print("Rejected patch binaries and metadata are included for audit/recovery.")
